# Brain CT Hemorrhage Detection Using Machine Learning

## 1. Project Overview

Brain hemorrhage, also known as intracranial hemorrhage, is a serious medical condition caused by bleeding inside the skull.

Computed Tomography (CT) scans are commonly used to detect brain hemorrhage because they can reveal bleeding areas inside the brain.

The goal of this project is to develop a Machine Learning model that analyzes brain CT scan images and classifies them into:

- **Normal:** CT scan without hemorrhage.
- **Hemorrhage:** CT scan containing hemorrhage.

## 2. Project Objectives

- Collect and understand a Brain CT Hemorrhage dataset.
- Perform Exploratory Data Analysis (EDA).
- Analyze image characteristics and class distribution.
- Apply image preprocessing techniques.
- Extract relevant features from CT images.
- Train different Machine Learning models.
- Compare model performance using evaluation metrics.
- Explore Deep Learning techniques such as CNN.

## 3. Technologies and Libraries

- Python
- NumPy
- Pandas
- Matplotlib
- Seaborn
- OpenCV
- Scikit-learn
- TensorFlow / Keras

## 4. Problem Type

**Binary Image Classification**

The model aims to distinguish between Normal and Hemorrhage brain CT images.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import pydicom
import sklearn
import os

print("All libraries imported successfully!")


In [ ]:
import zipfile
import os

zip_path = "archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    files = zip_ref.namelist()

    print("Total files:", len(files))

    for file in files[:20]:
        print(file)

In [ ]:
import zipfile
import os

zip_path = "archive.zip"

extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
image_path = "dataset/head_ct/head_ct"

images = os.listdir(image_path)

print("Total images:", len(images))
print("First 10 images:", images[:10])

## 5. Image Inspection

In this section, we will inspect the dataset images to understand their file formats, dimensions, and image characteristics before performing data analysis.

In [ ]:
import os

image_files = [
    file for file in os.listdir(image_path)
    if file.lower().endswith(('.png', '.jpg', '.jpeg'))
]

print("Number of images:", len(image_files))
print("Image format:", image_files[0].split('.')[-1])

In [ ]:
first_image_path = os.path.join(image_path, image_files[0])

image = cv2.imread(first_image_path)

print("Image shape:", image.shape)
print("Image data type:", image.dtype)

## Visualize the First Image


In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))

plt.title("Brain CT Scan Sample Image")
plt.axis("off")

plt.show()

## 6. Dataset Files and Labels

In this section, we will inspect the dataset files to identify whether label information is available and how the images are organized.

In [ ]:
import zipfile
import os

zip_path = "archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()

non_image_files = [
    file for file in all_files
    if not file.lower().endswith(('.png', '.jpg', '.jpeg'))
]

print("Total files:", len(all_files))
print("Non-image files:")
for file in non_image_files:
    print(file)

## 6. Dataset Files Inspection

In this section, we will inspect all files included in the dataset to identify whether label information or annotation files are available.

In [ ]:
import zipfile
import os

zip_path = "archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()

non_image_files = [
    file for file in all_files
    if not file.lower().endswith(('.png', '.jpg', '.jpeg'))
]

print("Non-image files:")
for file in non_image_files:
    print(file)

## 7. Loading Dataset Labels

In this section, we will load the labels.csv file using Pandas and examine its structure to understand how image labels are stored.

In [ ]:
labels_path = "dataset/labels.csv"

df = pd.read_csv(labels_path)

print("Dataset shape:", df.shape)
print("\nFirst 10 rows:")
display(df.head(10))

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

In [ ]:
class_counts = df['hemorrhage'].value_counts().sort_index()

print(class_counts)

In [ ]:
class_percentages = df['hemorrhage'].value_counts(normalize=True).sort_index() * 100

print(class_percentages)

In [ ]:
plt.figure(figsize=(7,5))

class_counts.plot(kind='bar')

plt.title("Class Distribution ")
plt.xlabel("Hemorrhange (0 = Normal, 1 = Hemorrhange)")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)

plt.show()

## 10. Understanding Image Representation

Digital images are represented as numerical arrays of pixels.
Each pixel contains an intensity value that represents the brightness of that pixel.

In this section, we will inspect the dimensions and numerical properties of our CT images.

In [ ]:
first_image_path = os.path.join(image_path, sorted(image_files)[0])

image = cv2.imread(first_image_path)

print("Image Shape:", image.shape)
print("Image Data Type:", image.dtype)
print("Minimum Pixel Value:", image.min())
print("Maximum Pixel Value:", image.max())

## 11. Grayscale Image Conversion

Grayscale conversion transforms a color image into a single-channel image.
Each pixel represents an intensity value ranging from black to white.

This reduces the number of image channels while preserving intensity information.

In [ ]:
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print("Original Image Shape:", image.shape)
print("Grayscale Image Shape:", gray_image.shape)

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray_image, cmap='gray')
plt.title("Grayscale Image")
plt.axis("off")

plt.show()

## 12. Pixel Intensity Distribution

Pixel intensity represents the brightness value of each pixel in a grayscale image.

A histogram shows the frequency distribution of pixel intensity values.

It helps us understand the brightness and contrast characteristics of CT images.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(gray_image.ravel(), bins=256, range=(0, 256))

plt.title("Pixel Intensity Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")

plt.show()

## 13. Comparing Normal and Hemorrhage Images

Before comparing images from different classes, we need to verify the relationship between image IDs, filenames, and their corresponding labels.

In [ ]:
print("First 10 labels:")
display(df.head(10))

print("\nLast 10 labels:")
display(df.tail(10))

print("\nImage filenames:")
print(sorted(image_files)[:10])

In [ ]:
df['id'] = df['id'].astype(int)

df['filename'] = df['id'].apply(lambda x: f"{x:03d}.png")

display(df.head(10))

In [ ]:
available_images = set(image_files)

df['image_exists'] = df['filename'].isin(available_images)

print("Images found:", df['image_exists'].sum())
print("Images missing:", (~df['image_exists']).sum())

In [ ]:
normal_row = df[df['hemorrhage'] == 0].iloc[0]
hemorrhage_row = df[df['hemorrhage'] == 1].iloc[0]

print("Normal Image:", normal_row['filename'])
print("Hemorrhage Image:", hemorrhage_row['filename'])

In [ ]:
normal_image = cv2.imread(
    os.path.join(image_path, normal_row['filename']),
    cv2.IMREAD_GRAYSCALE
)

hemorrhage_image = cv2.imread(
    os.path.join(image_path, hemorrhage_row['filename']),
    cv2.IMREAD_GRAYSCALE
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(normal_image, cmap='gray')
plt.title("Normal Image (Label 0)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(hemorrhage_image, cmap='gray')
plt.title("Hemorrhage Image (Label 1)")
plt.axis("off")

plt.show()

## 14. Histogram Comparison

In this section, we compare the pixel intensity distributions of a normal CT image and a hemorrhage CT image.

The comparison helps us investigate differences in image brightness and intensity distribution between the two classes.

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    normal_image.ravel(),
    bins=256,
    range=(0, 256),
    alpha=0.5,
    label="Normal"
)

plt.hist(
    hemorrhage_image.ravel(),
    bins=256,
    range=(0, 256),
    alpha=0.5,
    label="Hemorrhage"
)

plt.title("Normal vs Hemorrhage - Pixel Intensity Distribution")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.legend()

plt.show()

In [ ]:
print("Normal Image Statistics")
print("Mean:", np.mean(normal_image))
print("Standard Deviation:", np.std(normal_image))
print("Minimum:", np.min(normal_image))
print("Maximum:", np.max(normal_image))

print("\nHemorrhage Image Statistics")
print("Mean:", np.mean(hemorrhage_image))
print("Standard Deviation:", np.std(hemorrhage_image))
print("Minimum:", np.min(hemorrhage_image))
print("Maximum:", np.max(hemorrhage_image))

## 15. Statistical Analysis of the Entire Dataset

In this section, we extract statistical features from all CT images.

For each image, we calculate the mean, standard deviation, minimum, and maximum pixel intensity.

These features will help us analyze and compare the two image classes.

In [ ]:
image_features = []

for _, row in df.iterrows():

    filename = row['filename']
    label = row['hemorrhage']

    path = os.path.join(image_path, filename)

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        image_features.append({
            'filename': filename,
            'hemorrhage': label,
            'mean_intensity': np.mean(img),
            'std_intensity': np.std(img),
            'min_intensity': np.min(img),
            'max_intensity': np.max(img)
        })

features_df = pd.DataFrame(image_features)

print("Features Dataset Shape:", features_df.shape)

display(features_df.head())

In [ ]:
class_statistics = features_df.groupby('hemorrhage')[
    ['mean_intensity', 'std_intensity']
].mean()

display(class_statistics)

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=features_df,
    x='hemorrhage',
    y='mean_intensity'
)

plt.title("Mean Intensity Distribution by Class")
plt.xlabel("Class (0 = Normal, 1 = Hemorrhage)")
plt.ylabel("Mean Pixel Intensity")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=features_df,
    x='hemorrhage',
    y='std_intensity'
)

plt.title("Standard Deviation Distribution by Class")
plt.xlabel("Class (0 = Normal, 1 = Hemorrhage)")
plt.ylabel("Standard Deviation")

plt.show()


## 16. Correlation Analysis

Correlation analysis measures the strength and direction of the linear relationship between numerical variables.

We will use a correlation matrix to explore relationships between extracted image features and class labels.

In [ ]:
correlation_matrix = features_df[
    ['hemorrhage', 'mean_intensity', 'std_intensity',
     'min_intensity', 'max_intensity']
].corr()

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix of Image Features")
plt.show()

In [ ]:
print("Features Dataset Shape:", features_df.shape)
print("\nImages per Class:")
print(features_df['hemorrhage'].value_counts())
print(features_df.isnull().sum())

In [ ]:
class_counts = features_df['hemorrhage'].value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)

plt.title("Class Distribution")
plt.xlabel("Class (0 = Normal, 1 = Hemorrhage)")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
image_sizes = []

for filename in image_files:
    path = os.path.join(image_path, filename)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        image_sizes.append(img.shape)

print("Unique Image Sizes:")
print(set(image_sizes))

In [ ]:
import os

print("Current Directory:", os.getcwd())
print("Dataset Exists:", os.path.exists("dataset"))
print("Image Folder Exists:", os.path.exists(image_path))

In [ ]:
# Read one image
sample_path = os.path.join(image_path, image_files[0])
original_image = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)

# Get original dimensions
height, width = original_image.shape

# Target size
target_size = 128

# Calculate scaling factor
scale = min(target_size / width, target_size / height)

# Calculate new dimensions
new_width = int(width * scale)
new_height = int(height * scale)

# Resize while preserving aspect ratio
resized = cv2.resize(original_image, (new_width, new_height))

# Create a black canvas
final_image = np.zeros((target_size, target_size), dtype=np.uint8)

# Center the resized image
x_offset = (target_size - new_width) // 2
y_offset = (target_size - new_height) // 2

final_image[
    y_offset:y_offset + new_height,
    x_offset:x_offset + new_width
] = resized

# Display comparison
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(original_image, cmap='gray')
plt.title(f"Original: {width} x {height}")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(final_image, cmap='gray')
plt.title("Resized: 128 x 128")
plt.axis('off')

plt.show()

print("Original Shape:", original_image.shape)
print("Final Shape:", final_image.shape)

In [ ]:
resized_images = []

target_size = 128

for filename in image_files:

    path = os.path.join(image_path, filename)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    height, width = img.shape

    scale = min(target_size / width, target_size / height)

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(img, (new_width, new_height))

    final_image = np.zeros((target_size, target_size), dtype=np.uint8)

    x_offset = (target_size - new_width) // 2
    y_offset = (target_size - new_height) // 2

    final_image[
        y_offset:y_offset + new_height,
        x_offset:x_offset + new_width
    ] = resized

    resized_images.append(final_image)

print("Total Processed Images:", len(resized_images))
print("Image Shape:", resized_images[0].shape)

In [ ]:
import pandas as pd
import os
import numpy as np
import cv2

labels_path = "dataset/labels.csv"

df = pd.read_csv(labels_path)

df.columns = df.columns.str.strip()

df['id'] = df['id'].astype(int)

df['filename'] = df['id'].apply(lambda x: f"{x:03d}.png")

print(df.head())
print("Dataset Shape:", df.shape)

In [ ]:
resized_images = []
resized_labels = []
resized_filenames = []

target_size = 128

for _, row in df.iterrows():

    filename = row['filename']
    label = row['hemorrhage']

    path = os.path.join(image_path, filename)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    height, width = img.shape

    scale = min(target_size / width, target_size / height)

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(img, (new_width, new_height))

    final_image = np.zeros((target_size, target_size), dtype=np.uint8)

    x_offset = (target_size - new_width) // 2
    y_offset = (target_size - new_height) // 2

    final_image[
        y_offset:y_offset + new_height,
        x_offset:x_offset + new_width
    ] = resized

    resized_images.append(final_image)
    resized_labels.append(label)
    resized_filenames.append(filename)

print("Images:", len(resized_images))
print("Labels:", len(resized_labels))
print("Image Shape:", resized_images[0].shape)

In [ ]:
X = np.array(resized_images)
y = np.array(resized_labels)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("X Data Type:", X.dtype)
print("y Data Type:", y.dtype)

### Image Normalization

We applied Normalization to scale pixel values from **0–255 to 0–1**, making the data easier for the Machine Learning model to process and improving numerical stability during training.

In [ ]:
X_normalized = X.astype('float32') / 255.0

print("Original Data:")

print("Min:", X.min())
print("Max:", X.max())
print("Data Type:", X.dtype)

print("\nNormalized Data:")
print("Min:", X_normalized.min())
print("Max:", X_normalized.max())
print("Data Type:", X_normalized.dtype)

In [ ]:
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(X[0], cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(X_normalized[0], cmap='gray', vmin=0, vmax=1)
plt.title("Normalized Image")
plt.axis('off')

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train.shape)
print("Testing Images:", X_test.shape)

print("Training Labels:", y_train.shape)
print("Testing Labels:", y_test.shape)

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print("X_train_flat:", X_train_flat.shape)
print("X_test_flat:", X_test_flat.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_flat)

X_test_scaled = scaler.transform(X_test_flat)

print("Training Shape:", X_train_scaled.shape)
print("Testing Shape:", X_test_scaled.shape)

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

In [ ]:
#Prediction 
y_pred = model.predict(X_test_scaled)

print("Predicted Labels:")
print(y_pred)

print("\nActual Labels:")
print(y_test)

In [ ]:
y_prob = model.predict_proba(X_test_scaled)

print("Prediction Probabilities:")
print(y_prob[:10])

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

# Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Hemorrhage"]
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["Normal", "Hemorrhage"],
    yticklabels=["Normal", "Hemorrhage"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Find incorrect predictions
wrong_indices = np.where(y_test != y_pred)[0]

print("Total Misclassified Images:", len(wrong_indices))

for i in wrong_indices:
    print(
        "Index:", i,
        "| Actual:", y_test[i],
        "| Predicted:", y_pred[i]
    )

In [ ]:
plt.figure(figsize=(15, 8))

for plot_index, i in enumerate(wrong_indices):

    plt.subplot(2, 4, plot_index + 1)

    plt.imshow(X_test[i], cmap='gray')

    actual = "Normal" if y_test[i] == 0 else "Hemorrhage"
    predicted = "Normal" if y_pred[i] == 0 else "Hemorrhage"

    plt.title(
        f"Actual: {actual}\nPredicted: {predicted}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

## SVM (Support Vector Machine)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear')

svm_model.fit(X_train_scaled, y_train)

In [ ]:
svm_pred = svm_model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", svm_accuracy)
print("SVM Accuracy Percentage:", svm_accuracy * 100, "%")

In [ ]:
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Normal", "Hemorrhage"]
))

In [ ]:
print("Logistic Regression Accuracy:", accuracy)
print("SVM Accuracy:", svm_accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

svm_cm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    svm_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["Normal", "Hemorrhage"],
    yticklabels=["Normal", "Hemorrhage"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("SVM Confusion Matrix")
plt.show()

In [ ]:
svm_model = SVC(kernel='linear')

In [ ]:
from sklearn.svm import SVC

svm_rbf = SVC(kernel='rbf')

svm_rbf.fit(X_train_scaled, y_train)

rbf_pred = svm_rbf.predict(X_test_scaled)

## Evaluate RBF Kernel

## Calculate Accuracy

from sklearn.metrics import accuracy_score, classification_report

rbf_accuracy = accuracy_score(y_test, rbf_pred)

print("RBF SVM Accuracy:", rbf_accuracy)
print("RBF SVM Accuracy Percentage:", rbf_accuracy * 100, "%")

In [ ]:
## Classification Report

print(classification_report(
    y_test,
    rbf_pred,
    target_names=["Normal", "Hemorrhage"]
))

In [ ]:
## Compare all Models

print("Logistic Regression:", accuracy * 100, "%")
print("Linear SVM:", svm_accuracy * 100, "%")
print("RBF SVM:", rbf_accuracy * 100, "%")

## Hyperparameter Tuning using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train_scaled, y_train)

## Find the Best Parameters

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_search.best_score_)

print("\nBest Model:")
print(grid_search.best_estimator_)


## Test the Best SVM Model

In [ ]:
best_svm = grid_search.best_estimator_

best_pred = best_svm.predict(X_test_scaled)

best_accuracy = accuracy_score(y_test, best_pred)

print("Tuned SVM Accuracy:", best_accuracy)
print("Tuned SVM Accuracy Percentage:", best_accuracy * 100, "%")

In [ ]:
print(classification_report(
    y_test,
    best_pred,
    target_names=["Normal", "Hemorrhage"]
))

## Visualize the Tuned SVM Confusion Matrix

In [ ]:
tuned_cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    tuned_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["Normal", "Hemorrhage"],
    yticklabels=["Normal", "Hemorrhage"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Tuned RBF SVM Confusion Matrix")

plt.show()

In [ ]:
import joblib

joblib.dump(svm_model, 'brain_ct_svm_model.pkl')

joblib.dump(scaler, 'brain_ct_scaler.pkl')

print("Model and Scaler Saved Successfully!")

## Cross-Validation

We used 5-Fold Cross-Validation to evaluate the stability and generalization performance of the Linear SVM model across different training and validation splits.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    svm_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross-Validation Scores:", cv_scores)
print("Mean Accuracy:", cv_scores.mean())
print("Mean Accuracy Percentage:", cv_scores.mean() * 100, "%")

In [ ]:
fold_numbers = [1, 2, 3, 4, 5]

plt.figure(figsize=(8, 5))

plt.bar(fold_numbers, cv_scores * 100)

plt.axhline(
    y=cv_scores.mean() * 100,
    linestyle='--',
    label='Mean Accuracy'
)

plt.xlabel("Fold")
plt.ylabel("Accuracy (%)")
plt.title("5-Fold Cross-Validation Results")
plt.xticks(fold_numbers)
plt.legend()
plt.show()

## CNN

In [ ]:
X_cnn = X_normalized.reshape(-1, 128, 128, 1)

print("CNN Images Shape:", X_cnn.shape)
print("Labels Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train_cnn.shape)
print("Testing Images:", X_test_cnn.shape)

In [ ]:
%pip install tensorflow

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
%pip show tensorflow

In [ ]:
import tensorflow as tf

print(tf.__version__)

In [ ]:
%pip install --force-reinstall --no-cache-dir numpy==1.26.4

In [ ]:
import numpy as np

print("NumPy Version:", np.__version__)

In [ ]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)

In [ ]:
print("TensorFlow Version:", tf.__version__)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

In [ ]:
model_cnn = Sequential()

model_cnn.add(
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation='relu',
        input_shape=(128, 128, 1)
    )
)

In [ ]:
from tensorflow.keras.layers import Input

model_cnn = Sequential()

model_cnn.add(Input(shape=(128, 128, 1)))

model_cnn.add(
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation='relu'
    )
)

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.add(
    MaxPooling2D(pool_size=(2, 2))
)

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.add(
    Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation='relu'
    )
)

In [ ]:
model_cnn.add(
    MaxPooling2D(pool_size=(2, 2))
)

In [ ]:
model_cnn.add(Flatten())

In [ ]:
model_cnn.add(
    Dense(
        units=64,
        activation='relu'
    )
)

In [ ]:
model_cnn.add(
    Dropout(0.5)
)

In [ ]:
model_cnn.add(
    Dense(
        units=1,
        activation='sigmoid'
    )
)

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.layers

In [ ]:
model_cnn = Sequential()

model_cnn.add(Input(shape=(128, 128, 1)))

model_cnn.add(Conv2D(32, (3, 3), activation='relu'))
model_cnn.add(MaxPooling2D(pool_size=(2, 2)))

model_cnn.add(Conv2D(64, (3, 3), activation='relu'))
model_cnn.add(MaxPooling2D(pool_size=(2, 2)))

model_cnn.add(Flatten())

model_cnn.add(Dense(64, activation='relu'))

model_cnn.add(Dropout(0.5))

model_cnn.add(Dense(1, activation='sigmoid'))

In [ ]:
model_cnn.summary()


In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D 

In [ ]:
model_cnn = Sequential()

model_cnn.add(Input(shape=(128, 128, 1)))

model_cnn.add(Conv2D(32, (3, 3), activation='relu'))
model_cnn.add(MaxPooling2D(pool_size=(2, 2)))

model_cnn.add(Conv2D(64, (3, 3), activation='relu'))
model_cnn.add(MaxPooling2D(pool_size=(2, 2)))

model_cnn.add(GlobalAveragePooling2D())

model_cnn.add(Dense(64, activation='relu'))

model_cnn.add(Dropout(0.5))

model_cnn.add(Dense(1, activation='sigmoid'))

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model_cnn.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train_cnn.shape)
print("Testing Images:", X_test_cnn.shape)

In [ ]:
X_cnn = X_normalized.reshape(-1, 128, 128, 1)

print("CNN Images Shape:", X_cnn.shape)
print("Labels Shape:", y.shape)

In [ ]:
print("X exists:", 'X' in globals())
print("y exists:", 'y' in globals())
print("resized_images exists:", 'resized_images' in globals())
print("resized_labels exists:", 'resized_labels' in globals())

In [ ]:
X_normalized = X.astype('float32') / 255.0

print("Shape:", X_normalized.shape)
print("Minimum:", X_normalized.min())
print("Maximum:", X_normalized.max())

In [ ]:
X_cnn = X_normalized.reshape(-1, 128, 128, 1)

print("CNN Images Shape:", X_cnn.shape)
print("Labels Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train_cnn.shape)
print("Testing Images:", X_test_cnn.shape)

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model_cnn.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
print("Training Accuracy:", history.history['accuracy'])
print("Validation Accuracy:", history.history['val_accuracy'])

print("Training Loss:", history.history['loss'])
print("Validation Loss:", history.history['val_loss'])

print("Epochs Completed:", len(history.history['loss']))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Loss')
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_prob_cnn = model_cnn.predict(X_test_cnn)

y_pred_cnn = (y_prob_cnn >= 0.5).astype(int).ravel()

print("Confusion Matrix:")
print(confusion_matrix(y_test_cnn, y_pred_cnn))

print("\nClassification Report:")
print(classification_report(
    y_test_cnn,
    y_pred_cnn,
    target_names=["Normal", "Hemorrhage"]
))

## CNN Model Evaluation

The CNN model achieved a test accuracy of 70%.

The confusion matrix showed:
- 13 correctly classified Normal images.
- 15 correctly classified Hemorrhage images.
- 7 Normal images incorrectly classified as Hemorrhage.
- 5 Hemorrhage images incorrectly classified as Normal.

The model achieved a 75% recall for the Hemorrhage class
and a 68% precision.

Compared with the classical machine learning models,
the CNN achieved lower test accuracy than Linear SVM,
which achieved 85%.

The limited dataset size may have affected the CNN's
ability to learn meaningful image features.

## Data Augmentation

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    RandomRotation,
    RandomZoom,
    RandomTranslation
)

data_augmentation = Sequential([
    RandomRotation(0.03),
    RandomZoom(0.1),
    RandomTranslation(0.05, 0.05)
])

In [ ]:
model_cnn_aug = Sequential([
    Input(shape=(128, 128, 1)),

    data_augmentation,

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    GlobalAveragePooling2D(),

    Dense(64, activation='relu'),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

In [ ]:
model_cnn_aug.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_cnn_aug.summary()

In [ ]:
early_stop_aug = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history_aug = model_cnn_aug.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop_aug]
)

In [ ]:
print("Training Accuracy:", history_aug.history['accuracy'])
print("Validation Accuracy:", history_aug.history['val_accuracy'])

print("Training Loss:", history_aug.history['loss'])
print("Validation Loss:", history_aug.history['val_loss'])

print("Epochs Completed:", len(history_aug.history['loss']))

In [ ]:
import numpy as np

print("Training Labels Distribution:")
print(np.unique(y_train_cnn, return_counts=True))

print("\nValidation Labels Distribution:")
print(np.unique(
    y_train_cnn[:32],
    return_counts=True
))

In [ ]:
import matplotlib.pyplot as plt

sample_image = X_train_cnn[0]

augmented_image = data_augmentation(
    np.expand_dims(sample_image, axis=0),
    training=True
)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_image.squeeze(), cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(augmented_image[0].numpy().squeeze(), cmap='gray')
plt.title("Augmented Image")
plt.axis("off")

plt.show()

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(
    model_cnn,
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True,
    to_file="cnn_architecture.png"
)

In [ ]:
%pip install pydot graphviz

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nPydot location:")
!{sys.executable} -m pip show pydot

print("\nGraphviz location:")
!which dot

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout
)
from tensorflow.keras.optimizers import Adam

model_cnn_lr = Sequential([
    Input(shape=(128, 128, 1)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    GlobalAveragePooling2D(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

model_cnn_lr.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_cnn_lr.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop_lr = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history_lr = model_cnn_lr.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop_lr]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_lr.history['accuracy'], label='Training Accuracy')
plt.plot(history_lr.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Learning Rate: 0.0001 - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_lr.history['loss'], label='Training Loss')
plt.plot(history_lr.history['val_loss'], label='Validation Loss')
plt.title('CNN Learning Rate: 0.0001 - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss_lr, test_accuracy_lr = model_cnn_lr.evaluate(
    X_test_cnn,
    y_test_cnn
)

print("Test Loss:", test_loss_lr)
print("Test Accuracy:", test_accuracy_lr)
print("Test Accuracy Percentage:", test_accuracy_lr * 100, "%")

In [ ]:
import pandas as pd

final_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Linear SVM',
        'RBF SVM',
        'CNN',
        'CNN - Data Augmentation',
        'CNN - Learning Rate 0.0001'
    ],
    'Test Accuracy (%)': [
        82.5,
        85.0,
        82.5,
        70.0,
        None,
        50.0
    ]
})

final_comparison

In [ ]:
svm_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred_svm = svm_model.predict(X_test_scaled)

print("Final Classification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Normal", "Hemorrhage"]
))

cm = confusion_matrix(y_test, y_pred_svm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Hemorrhage"]
)

disp.plot(cmap="Blues")
plt.title("Linear SVM - Confusion Matrix")
plt.show()

In [ ]:
print("Original image shape:", X_train.shape)
print("Flattened training shape:", X_train_flat.shape)
print("Scaled training shape:", X_train_scaled.shape)

print("SVM model:", svm_model)

In [ ]:
%pip install ipywidgets pillow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from PIL import Image
from IPython.display import display, clear_output

upload = widgets.FileUpload(
    accept='.png,.jpg,.jpeg',
    multiple=False
)

output = widgets.Output()

def predict_image(change):

    with output:
        clear_output(wait=True)

        if not upload.value:
            return

        file_info = upload.value

        if isinstance(file_info, dict):
            file_info = list(file_info.values())[0]
            image_bytes = file_info['content']
        else:
            image_bytes = file_info[0]['content']

        # Read image
        image = Image.open(
            image_bytes if hasattr(image_bytes, 'read') else
            __import__('io').BytesIO(image_bytes)
        )

        # Preprocessing
        image = image.convert('L')
        image = image.resize((128, 128))

        image_array = np.array(image).astype('float32')

        # Flatten
        image_flat = image_array.reshape(1, -1)

        # Scaling
        image_scaled = scaler.transform(image_flat)

        # Prediction
        prediction = svm_model.predict(image_scaled)[0]

        # Display image and result
        plt.figure(figsize=(4, 4))
        plt.imshow(image_array, cmap='gray')
        plt.axis('off')
        plt.title("Uploaded CT Image")
        plt.show()

        if prediction == 1:
            print("Prediction: Hemorrhage")
        else:
            print("Prediction: Normal")

upload.observe(predict_image, names='value')

display(widgets.VBox([
    widgets.HTML("<h3>Brain CT Hemorrhage Detection</h3>"),
    upload,
    output
]))

In [ ]:
import ipywidgets as widgets
print(widgets.__version__)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.Button(description="Test Upload Interface"))


In [ ]:
%pip install -U jupyterlab_widgets ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.Button(description="Test Upload Interface"))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(
    accept='.png,.jpg,.jpeg',
    multiple=False
)

display(upload)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

# Get uploaded image
file_info = upload.value[0]

# Read image
image = Image.open(io.BytesIO(bytes(file_info['content'])))

# Convert to grayscale and resize
image = image.convert('L')
image = image.resize((128, 128))

# Convert to NumPy array
image_array = np.array(image).astype('float32')

# Flatten image
image_flat = image_array.reshape(1, -1)

# Scale using the same scaler used during training
image_scaled = scaler.transform(image_flat)

# Predict
prediction = svm_model.predict(image_scaled)[0]

# Display image
plt.figure(figsize=(4, 4))
plt.imshow(image_array, cmap='gray')
plt.axis('off')
plt.show()

print("Prediction:", "Hemorrhage" if prediction == 1 else "Normal")

In [ ]:
print([name for name in globals() if 'scal' in name.lower()])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_train_flat)

print("Scaler fitted successfully!")

In [ ]:
important_variables = [
    "X", "y", "X_train", "X_test",
    "X_train_flat", "X_train_scaled",
    "svm_model", "scaler"
]

for name in important_variables:
    print(name, ":", name in globals())

In [ ]:
print(X.shape)
print(svm_model)